In [1]:
import torch.nn as nn
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

False

In [3]:
train_df = pd.read_csv("data/asl_data/sign_mnist_train.csv")
valid_df = pd.read_csv("data/asl_data/sign_mnist_valid.csv")

In [4]:
sample_df = train_df.head().copy()
sample_df.pop('label')
sample_x = sample_df.values
sample_x

array([[107, 118, 127, ..., 204, 203, 202],
       [155, 157, 156, ..., 103, 135, 149],
       [187, 188, 188, ..., 195, 194, 195],
       [211, 211, 212, ..., 222, 229, 163],
       [164, 167, 170, ..., 163, 164, 179]], shape=(5, 784))

In [5]:
sample_x.shape

(5, 784)

In [6]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

sample_x = sample_x.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
sample_x.shape

(5, 1, 28, 28)

In [7]:
class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()  # Some operations below are in-place
        y_df = x_df.pop('label')
        x_df = x_df.values / 255  # Normalize values from 0 to 1
        x_df = x_df.reshape(-1, IMG_CHS, IMG_WIDTH, IMG_HEIGHT)
        self.xs = torch.tensor(x_df).float().to(device)
        self.ys = torch.tensor(y_df).to(device)

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)

In [8]:
BATCH_SIZE = 32

train_data = MyDataset(train_df)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
valid_N = len(valid_loader.dataset)

In [9]:
batch = next(iter(train_loader))
batch

[tensor([[[[0.6902, 0.6941, 0.6902,  ..., 0.6392, 0.6314, 0.6314],
           [0.6941, 0.6980, 0.6941,  ..., 0.6431, 0.6353, 0.6353],
           [0.6980, 0.6980, 0.6980,  ..., 0.6431, 0.6392, 0.6392],
           ...,
           [0.7412, 0.7412, 0.7451,  ..., 0.6902, 0.6863, 0.6863],
           [0.7412, 0.7451, 0.7451,  ..., 0.6941, 0.6902, 0.6863],
           [0.7412, 0.7451, 0.7373,  ..., 0.6980, 0.6941, 0.6863]]],
 
 
         [[[0.6706, 0.6706, 0.6706,  ..., 0.0275, 0.0431, 0.1137],
           [0.6784, 0.6784, 0.6824,  ..., 0.0510, 0.0784, 0.1843],
           [0.6941, 0.6941, 0.6980,  ..., 0.0706, 0.1490, 0.2549],
           ...,
           [0.4000, 0.3961, 0.3961,  ..., 0.8392, 0.5412, 0.6314],
           [0.4000, 0.4039, 0.4039,  ..., 0.7882, 0.4745, 0.6824],
           [0.4000, 0.4039, 0.4078,  ..., 0.7529, 0.4824, 0.6824]]],
 
 
         [[[0.7059, 0.7098, 0.7098,  ..., 0.6196, 0.6196, 0.6157],
           [0.7098, 0.7098, 0.7098,  ..., 0.6235, 0.6196, 0.6196],
           [0.7137

In [10]:
batch[0].shape

torch.Size([32, 1, 28, 28])

In [11]:
batch[1].shape

torch.Size([32])

In [12]:
n_classes = 24
kernel_size = 3
flattened_img_size = 75 * 3 * 3

model = nn.Sequential(
    # First convolution
    nn.Conv2d(IMG_CHS, 25, kernel_size, stride=1, padding=1),  # 25 x 28 x 28
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 25 x 14 x 14
    # Second convolution
    nn.Conv2d(25, 50, kernel_size, stride=1, padding=1),  # 50 x 14 x 14
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # 50 x 7 x 7
    # Third convolution
    nn.Conv2d(50, 75, kernel_size, stride=1, padding=1),  # 75 x 7 x 7
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 75 x 3 x 3
    # Flatten to Dense
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, n_classes)
)

In [13]:
model = model.to(device)
model

Sequential(
  (0): Conv2d(1, 25, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(25, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (4): Conv2d(25, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Dropout(p=0.2, inplace=False)
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Conv2d(50, 75, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (10): BatchNorm2d(75, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): ReLU()
  (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (13): Flatten(start_dim=1, end_dim=-1)
  (14): Linear(in_features=675, out_features=512, bias=True)
  (15): Dropout(p=0.3, inplace=False)
  (16): ReLU()
  (17): Linear(in_features

In [14]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters())

In [15]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

In [16]:
def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [17]:
def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

In [18]:
epochs = 20

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

Epoch: 0
Train - Loss: 260.0156 Accuracy: 0.9117
Valid - Loss: 18.9575 Accuracy: 0.9697
Epoch: 1
Train - Loss: 20.3897 Accuracy: 0.9939
Valid - Loss: 22.8492 Accuracy: 0.9663
Epoch: 2
Train - Loss: 16.4020 Accuracy: 0.9944
Valid - Loss: 34.8116 Accuracy: 0.9525
Epoch: 3
Train - Loss: 5.8037 Accuracy: 0.9982
Valid - Loss: 13.5455 Accuracy: 0.9735
Epoch: 4
Train - Loss: 10.0981 Accuracy: 0.9964
Valid - Loss: 33.0925 Accuracy: 0.9576
Epoch: 5
Train - Loss: 5.1311 Accuracy: 0.9983
Valid - Loss: 19.3830 Accuracy: 0.9647
Epoch: 6
Train - Loss: 10.8912 Accuracy: 0.9965
Valid - Loss: 33.2655 Accuracy: 0.9504
Epoch: 7
Train - Loss: 6.6685 Accuracy: 0.9981
Valid - Loss: 8.2310 Accuracy: 0.9887
Epoch: 8
Train - Loss: 7.0773 Accuracy: 0.9976
Valid - Loss: 28.6163 Accuracy: 0.9559
Epoch: 9
Train - Loss: 1.3407 Accuracy: 0.9996
Valid - Loss: 23.4807 Accuracy: 0.9710
Epoch: 10
Train - Loss: 8.3953 Accuracy: 0.9971
Valid - Loss: 8.5559 Accuracy: 0.9875
Epoch: 11
Train - Loss: 1.1447 Accuracy: 0.9996
V